# Workitem Signing with APS Automation SDK

This example shows a complete and deterministic signing preparation flow for APS Design Automation public activities.

Flow in this notebook:
1. Generate RSA key files.
2. Set nickname first.
3. Upload public key to `forgeapps/me`.
4. Create AppBundle.
5. Create Activity.
6. Sign that created activity ID.

This notebook ends after signing the activity.


## 1) Setup

This cell loads credentials from `.env` (or environment variables).

Expected variables:
- `CLIENT_ID`
- `CLIENT_SECRET`
- optional `APS_NICKNAME`

If required values are missing, the notebook fails early.


In [ ]:
import json
import os
import uuid
from pathlib import Path

from dotenv import load_dotenv

from aps_automation_sdk import (
    Activity,
    ActivityInputParameter,
    ActivityJsonParameter,
    ActivityOutputParameter,
    AppBundle,
    export_public_key,
    generate_key_file,
    get_token,
    set_nickname,
    sign_activity,
    upload_public_key,
    delete_activity,
    delete_appbundle,
)

load_dotenv()

CLIENT_ID = os.getenv("CLIENT_ID")
CLIENT_SECRET = os.getenv("CLIENT_SECRET")
APS_NICKNAME = "myUniqueNick_123_123" # Change this

KEY_DIR = "keys"
PRIVATE_KEY_PATH = f"{KEY_DIR}/mykey.json"
PUBLIC_KEY_PATH = f"{KEY_DIR}/mypublickey.json"
os.makedirs(KEY_DIR, exist_ok=True)

if not CLIENT_ID or not CLIENT_SECRET:
    raise ValueError("Missing CLIENT_ID or CLIENT_SECRET in environment/.env")



## 2) Generate private key and export public key

This creates two files (paths are explicit and customizable):
- `PRIVATE_KEY_PATH` (default `keys/mykey.json`): private key material (keep secret, never commit)
- `PUBLIC_KEY_PATH` (default `keys/mypublickey.json`): public key APS stores for verification


In [ ]:
saved_private = generate_key_file(PRIVATE_KEY_PATH)
export_public_key(PRIVATE_KEY_PATH, PUBLIC_KEY_PATH)

print(f"Generated private key: {saved_private}")
print(f"Generated public key: {os.path.abspath(PUBLIC_KEY_PATH)}")


## 3) Set nickname first

Create the nickname before creating AppBundles or Activities.

APS guidance for this flow:
- set nickname first, or set nickname + public key in the same `PATCH /forgeapps/me` call,
- do not create AppBundles/Activities first and plan to set nickname later,
- if the app already has Design Automation data, nickname assignment may require deleting existing DA data first.

Reference: https://aps.autodesk.com/en/docs/design-automation/v3/reference/http/forgeapps-me-PATCH/


In [ ]:
token = get_token(CLIENT_ID, CLIENT_SECRET)

if not APS_NICKNAME:
    raise ValueError("Missing APS_NICKNAME in environment/.env for nickname-first flow")

nickname = set_nickname(token=token, nickname=APS_NICKNAME)
print(f"Nickname set/confirmed: {nickname}")


## 4) Upload public key to `forgeapps/me` (US-East)

After nickname is set, upload the public key for APS signature verification.

Sequence reminder:
1. Set nickname.
2. Upload public key.
3. Create AppBundle.
4. Create Activity.
5. Sign activity.


In [ ]:
with open(PUBLIC_KEY_PATH, "r", encoding="utf-8") as f:
    public_key = json.load(f)

profile = upload_public_key(token=token, public_key=public_key)

print("Uploaded public key profile:")
print(json.dumps(profile, indent=2))


## 5) Create AppBundle and Activity using `examples/Revit_02_export_to_ifc`

This section reuses the compiled bundle from:
- `examples/Revit_02_export_to_ifc/files/IfcExportDA.bundle.zip`

Paths are resolved from the repository root so the notebook works when launched from different working directories.


In [ ]:
cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / "pyproject.toml").exists()), None)
if repo_root is None:
    raise RuntimeError("Could not resolve repository root (pyproject.toml not found in cwd parents)")

bundle_zip_path = repo_root / "examples" / "Revit_02_export_to_ifc" / "files" / "IfcExportDA.bundle.zip"
if not bundle_zip_path.exists():
    raise FileNotFoundError(f"Bundle not found: {bundle_zip_path}")

suffix = uuid.uuid4().hex[:8]
alias = "prod"
engine = "Autodesk.Revit+2024"
app_bundle_name = f"IfcExportBundleSign{suffix}"
activity_name = f"RevitIfcExportSign{suffix}"

print(f"Repo root: {repo_root}")
print(f"Bundle path: {bundle_zip_path}")
print(f"AppBundle ID: {app_bundle_name}")
print(f"Activity ID: {activity_name}")


In [ ]:
bundle = AppBundle(
    appBundleId=app_bundle_name,
    engine=engine,
    alias=alias,
    zip_path=str(bundle_zip_path),
    description="IFC export bundle for signing flow example",
)

bundle.deploy(token)
appbundle_full_alias = f"{nickname}.{app_bundle_name}+{alias}"

input_revit = ActivityInputParameter(
    name="rvtFile",
    localName="input.rvt",
    verb="get",
    description="Input Revit File",
    required=True,
    is_engine_input=True,
)
output_zip = ActivityOutputParameter(
    name="result",
    localName="result",
    verb="put",
    description="Zipped IFC output",
    zip=True,
)
input_json = ActivityJsonParameter(
    name="ifcSettings",
    localName="ifc_settings.json",
    verb="get",
    description="Inline IFC settings JSON",
)

activity = Activity(
    id=activity_name,
    parameters=[input_revit, output_zip, input_json],
    engine=engine,
    appbundle_full_name=appbundle_full_alias,
    description="Signing flow example activity",
    alias=alias,
)
activity.set_revit_command_line()
activity.deploy(token=token)

activity_full_alias = f"{nickname}.{activity_name}+{alias}"
print(f"AppBundle deployed: {appbundle_full_alias}")
print(f"Activity deployed: {activity_full_alias}")


## 6) Sign the created activity ID (end of test)

This signs the exact activity alias created above and ends the test flow.


In [ ]:
signature = sign_activity(PRIVATE_KEY_PATH, activity_full_alias)
print("Activity ID:", activity_full_alias)
print("Signature:", signature)


## Optional CLI parity

Equivalent CLI commands:

```bash
aps-automation signing generate --keyfile keys/mykey.json
aps-automation signing export --keyfile keys/mykey.json --pubkeyfile keys/mypublickey.json
aps-automation public-key upload --pubkeyfile keys/mypublickey.json
aps-automation signing sign --keyfile keys/mykey.json --activity-id "yourNickname.YourActivity+prod"
```


## 7) Cleanup (delete Activity/AppBundle and local keys)

This final step cleans up remote Design Automation resources created in this notebook and removes local key files from `keys/`.


In [ ]:
print("Starting cleanup...")

try:
    delete_activity(activityId=activity_name, token=token)
    print(f"Deleted activity: {activity_full_alias}")
except Exception as e:
    print(f"Could not delete activity (may not exist): {e}")

try:
    delete_appbundle(appbundleId=app_bundle_name, token=token)
    print(f"Deleted appbundle: {app_bundle_name}")
except Exception as e:
    print(f"Could not delete appbundle (may not exist): {e}")

for key_path in [PRIVATE_KEY_PATH, PUBLIC_KEY_PATH]:
    path_obj = Path(key_path)
    if path_obj.exists():
        path_obj.unlink()
        print(f"Deleted key file: {path_obj}")
    else:
        print(f"Key file not found: {path_obj}")

key_dir = Path(KEY_DIR)
if key_dir.exists() and not any(key_dir.iterdir()):
    key_dir.rmdir()
    print(f"Deleted empty key folder: {key_dir}")
elif key_dir.exists():
    print(f"Key folder not empty, left in place: {key_dir}")
